# Machine Learning for Single-Cell Biology
***

Winter Semester 2025/26<br />
Manfred Claassen, Maximilian Wirth, Jan Schleicher
<br />
***

## Assignment 1

In [1]:
import numpy as np
import pandas as pd
from scipy import stats, io, sparse
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from statsmodels.stats.multitest import multipletests

import os
from utils import import_mtx_expression_data
from typing import Tuple

from matplotlib import pyplot as plt
import seaborn as sns

sns.set_style("ticks")

%matplotlib inline

In [2]:
DATA_PATH = "data/"
OUTPUT_PATH = "output/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [3]:
# sparse matrix
def import_mtx_expression_data(mtx_file, barcodes_file, genes_file):
    matrix = io.mmread(mtx_file).tocsr().astype(np.float32)
    barcodes = pd.read_csv(barcodes_file, header=None)[0]
    genes = pd.read_csv(genes_file, header=None)[0]
    return pd.DataFrame.sparse.from_spmatrix(matrix, index=barcodes, columns=genes)
def to_sparse(df):
    return df.sparse.to_coo().tocsr().astype(np.float32)
def normalize(X):
    total = np.asarray(X.sum(axis=1)).ravel() #normalizes each cell to 10 000 total counts
    X = X.multiply((10000/np.maximum(total, 1))[:, None]).tocsr()
    X.data = np.log1p(X.data)
    return X

### Task 1

In [4]:
# Load the data
expression_data_1 = import_mtx_expression_data(os.path.join(DATA_PATH, "expression_data_1.mtx"),
                                               os.path.join(DATA_PATH, "expression_data_1_barcodes.tsv"),
                                               os.path.join(DATA_PATH, "expression_data_1_genes.tsv"))
metadata_1 = pd.read_csv(os.path.join(DATA_PATH, "expression_data_1_metadata.tsv"), sep="\t", index_col=0)

# If required, perform preprocessing of the data
#########################
# INSERT YOUR CODE HERE #
#########################
expression_data_1 = import_mtx_expression_data(os.path.join(DATA_PATH, "expression_data_1.mtx"), os.path.join(DATA_PATH, "expression_data_1_barcodes.tsv"), os.path.join(DATA_PATH, "expression_data_1_genes.tsv"))

metadata_1 = pd.read_csv(os.path.join(DATA_PATH, "expression_data_1_metadata.tsv"), sep="\t", index_col=0)
metadata_1 = metadata_1.reindex(expression_data_1.index) # allignment of metadata with expression data
assert metadata_1["new_pheno"].notna().all()
assert metadata_1["new_sampleID"].notna().all()

print("expression data:", expression_data_1.shape)
print("metadata:", metadata_1.shape)

expression data: (19302, 16254)
metadata: (19302, 2)


**1A** - we have to identify the five anonymous cell phenotypes T1–T5 and the six anonymous organs A–F.

Steps:
- We first calculate the proportion of each pheno in each organ and compare these proportions with Figure 2. According to the paper:
- advanced exhausted cells are particularly common in the bone marrow and liver.
- effect like cells are especially common in the lungs.
- memory-like cells are mainly found in spleen and lymph nodes.
- proliferating cells have cell-cycle expression signature.
- intermediate cells have characteristics of both effector-like and exhausted cells.

- After we identify the groups based on their proportions, we also check the expression of characteristic marker genes for Figure 2 and 3 (both are important because pheno proportions alone may not be enough for distinguishing similar groups)

In [7]:
def task1a_identification(expression_data: pd.DataFrame, metadata: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Analyze the data to identify cell types and organs
    :param expression_data: expression matrix of shape (n_cells, n_genes)
    :param metadata: metadata containing cell groupings and sample IDs
    :return: organ and cell type identities
    """
    organ_identity = {"A": "", "B": "", "C": "", "D": "", "E": "", "F": ""}
    cell_type_identity = {"T1": "", "T2": "", "T3": "", "T4": "", "T5": ""}

    #########################
    # INSERT YOUR CODE HERE #
    #########################
# proportion of each pheno in every organ
    proportions = pd.crosstab(metadata["new_pheno"], metadata["new_sampleID"], normalize="columns")*100
    print("cell type proportions:")
    print(proportions.round(2))
    # we identiy the phenotypes by using the figure
    Exh = proportions.sum(axis=1).idxmax() # highest overall proportion
    E = proportions.sum(axis=1).nlargest(2).index[1] # 2nd
    M = proportions.min(axis=1).idxmin() # lowest min proportion
    # we separate I and P using the organ where M is highest
    remaining = [p for p in proportions.index
                 if p not in [Exh, E, M]]
    memory_organ = proportions.loc[M].idxmax()
    I = proportions.loc[remaining, memory_organ].idxmax()
    P = [p for p in remaining if p != I][0]
    cell_type_identity[Exh] = 'Exh'
    cell_type_identity[E] = 'E'
    cell_type_identity[M] = 'M'
    cell_type_identity[I] = 'I'
    cell_type_identity[P] = 'P'
    # identify organs by their pheno proportions
    liver = proportions.loc[Exh].idxmax()
    BM = proportions.loc[Exh].nlargest(2).index[1]
    lung = proportions.loc[E].idxmax()
    LN = proportions.loc[M].idxmax()
    spleen = proportions.loc[M].nlargest(2).index[1]

    blood = [organ for organ in proportions.columns
             if organ not in [liver, BM, lung, LN, spleen]][0]
    organ_identity[liver] = 'liver'
    organ_identity[BM] = "BM"
    organ_identity[lung] = "lung"
    organ_identity[LN] = "LN"
    organ_identity[spleen] = "spleen"
    organ_identity[blood] = "blood"

    markers = ["Cx3cr1", "Gzmb", "Tbx21", "Zeb2", "Pdcd1", "Lag3", "Cd160", "Cxcr6","Tcf7", "Id3", "Ccr7", "Sell", "Mki67", "Birc5", "Cks2", "Nr4a1"]

    markers = [gene for gene in markers if gene in expression_data.columns]
    counts = to_sparse(expression_data)
    total = np.asarray(counts.sum(axis=1)).ravel()

    if markers:
        indices = expression_data.columns.get_indexer(markers)
        marker_values = counts[:, indices].toarray()

        marker_values = np.log1p(marker_values / np.maximum(total[:, None], 1) * 10000)
        marker_data = pd.DataFrame(marker_values, columns=markers)
        marker_data["phenotype"] = metadata["new_pheno"].to_numpy()
        marker_means = marker_data.groupby("phenotype").mean()

        print("mean expression of marker genes:")
        print(marker_means.round(2))
        #check tissue specific marker genes
        tissue_markers = ["Ngp", "Camp", "S100a8","S100a9", "Nr4a1", "Jun"]
    tissue_markers = [gene for gene in tissue_markers if gene in expression_data.columns]

    if tissue_markers:
        indices = expression_data.columns.get_indexer(tissue_markers)
        values = counts[:, indices].toarray()
        values = np.log1p(values / np.maximum(total[:, None], 1) * 10000)

        tissue_data = pd.DataFrame(values, columns=tissue_markers)
        tissue_data["organ"] = metadata["new_sampleID"].to_numpy()

        print("tissue marker expression:")
        print(tissue_data.groupby("organ").mean().round(2))

    return pd.DataFrame.from_dict(organ_identity, orient="index"),\
           pd.DataFrame.from_dict(cell_type_identity, orient="index")

In [8]:
task1a_organ_identity, task1a_cell_type_identity = task1a_identification(expression_data_1, metadata_1)

task1a_organ_identity.to_csv(os.path.join(OUTPUT_PATH, "task1a_organ_identity.txt"), sep="\t", header=False)
task1a_cell_type_identity.to_csv(os.path.join(OUTPUT_PATH, "task1a_cell_type_identity.txt"), sep="\t", header=False)

display(task1a_organ_identity)
display(task1a_cell_type_identity)

cell type proportions:
new_sampleID      A      B      C      D      E      F
new_pheno                                             
T1            24.50   2.75   3.56  37.17   4.14  61.98
T2            56.15  49.67  86.91  50.90  91.23  18.75
T3             5.45  19.55   2.31   5.85   2.83  10.08
T4            11.16  15.18   0.38   0.18   0.17   1.20
T5             2.74  12.85   6.84   5.89   1.63   7.99
mean expression of marker genes:
           Cx3cr1  Gzmb  Tbx21  Zeb2  Pdcd1  Lag3  Cd160  Cxcr6  Tcf7   Id3  \
phenotype                                                                     
T1           0.90  2.61   0.63  1.11   0.84  1.16   0.77   1.47  0.08  0.04   
T2           0.34  2.33   0.43  0.47   1.13  1.33   1.23   2.01  0.06  0.04   
T3           0.63  2.56   0.49  0.75   0.97  1.38   0.85   1.65  0.11  0.07   
T4           0.50  2.39   0.48  0.60   0.99  1.46   0.84   1.68  0.18  0.12   
T5           0.54  2.51   0.48  0.69   1.07  1.38   0.96   1.77  0.10  0.06   

     

,0
A,spleen
B,LN
C,BM
D,blood
E,liver
F,lung


,0
T1,E
T2,Exh
T3,I
T4,M
T5,P



**1B** - we need to predict the phenotype of each cell in dataset 2

- We use dataset 1 as reference data because the cell phenotypes were identified in 1A. The idea is that cells belonging to the same phenotype should have similar gene expression profiles.
- We calculate an average expression profile for each phenotype and compare new cells with these reference profiles using Pearson correlation. Before calculating the correlations, we normalize gene expression to account for differences in sequencing depth. We also select genes that differ between phenotypes, because these genes are more useful for classification.
- To check whether the method works, we divide the reference cells into training and test sets. We then report accuracy and F1 score before applying the classifier to the new dataset.


In [39]:
# Load the data
expression_data_2 = import_mtx_expression_data(os.path.join(DATA_PATH, "expression_data_2.mtx"),
                                               os.path.join(DATA_PATH, "expression_data_2_barcodes.tsv"),
                                               os.path.join(DATA_PATH, "expression_data_2_genes.tsv"))

# If required, perform preprocessing of the data
#########################
# INSERT YOUR CODE HERE #
#########################
print("dataset 2:", expression_data_2.shape)

dataset 2: (4826, 16254)


In [57]:
def task1b_annotation(expression_data: pd.DataFrame, expression_data_ref: pd.DataFrame, metadata_ref) -> pd.DataFrame:
    """
    Annotate each cell with its corresponding cell type
    :param expression_data: expression matrix of shape (n_cells, n_genes)
    :param expression_data_ref: reference expression matrix of shape (n_cells_ref, n_genes)
    :param metadata_ref: reference metadata with n_cells_ref rows and a column containing true cell types for expression_data_ref
    :return: cell type annotations for expression_data
    """
    from sklearn.svm import LinearSVC
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.feature_selection import SelectKBest, chi2
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, classification_report)
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

    cell_annotations = pd.DataFrame(index=expression_data.index.values, columns=["phenotype"])

    #########################
    # INSERT YOUR CODE HERE #
    #########################

    #we are using the labels identified at A1
    phenotype_map = task1a_cell_type_identity.iloc[:, 0]
    labels = metadata_ref["new_pheno"].map(phenotype_map)
    labels = labels.reindex(expression_data_ref.index)
    assert labels.notna().all()

    # we use the genes common between the two datasets
    common_genes = expression_data_ref.columns.intersection(expression_data.columns)

    X_ref = to_sparse(expression_data_ref[common_genes])
    X = to_sparse(expression_data[common_genes])

    X_ref = normalize(X_ref)
    X = normalize(X)

    y = labels.to_numpy()
    phenotypes = np.array(["E", "Exh", "I", "M", "P"])

    # split for validation
    train_idx, test_idx = train_test_split(
        np.arange(X_ref.shape[0]),
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # select the most relevant genes
    # selector = SelectKBest(chi2, k=min(1000, X_ref.shape[1]))
    # selector = SelectKBest(chi2, k=min(2000, X_ref.shape[1]))
    # X_train = selector.fit_transform(X_ref[train_idx], y[train_idx])
    # X_test = selector.transform(X_ref[test_idx])

    # genes associated with the phenotypes in the paper
    markers = [
        "Tcf7", "Id3", "Ccr7", "Sell", "Slamf6", "Il7r",
        "Cx3cr1", "Gzmb", "Prf1", "Zeb2", "Tbx21", "Klf2",
        "Pdcd1", "Lag3", "Cd160", "Cd244", "Eomes", "Cxcr6", "Tox",
        "Mki67", "Birc5", "Cks2", "Mcm2", "Mcm3"
    ]

    # select additional genes using the training data
    # selector = SelectKBest(chi2, k=min(200, X_ref.shape[1]))
    # selector.fit(X_ref[train_idx], y[train_idx])
    #
    # selected_genes = list(common_genes[selector.get_support()])

    # old gene selection using chi2
    # selected_genes = []
    #
    # for p in phenotypes:
    #     binary_labels = (y[train_idx] == p).astype(int)
    #
    #     scores, _ = chi2(X_ref[train_idx], binary_labels)
    #     scores = np.nan_to_num(scores)
    #
    #     best = np.argsort(scores)[-50:]
    #
    #     for gene in common_genes[best]:
    #         if gene not in selected_genes:
    #             selected_genes.append(gene)

    # select genes with higher expression in each phenotype
    def select_genes(X_train, y_train):
        selected_genes = []

        for p in phenotypes:
            cells_p = X_train[y_train == p]
            other_cells = X_train[y_train != p]

            mean_p = np.asarray(cells_p.mean(axis=0)).ravel()
            mean_other = np.asarray(other_cells.mean(axis=0)).ravel()

            differences = mean_p - mean_other

            best = np.argsort(differences)[-50:]

            for index in best:
                if differences[index] > 0:
                    gene = common_genes[index]

                    if gene not in selected_genes:
                        selected_genes.append(gene)

        # also keep the marker genes from the paper
        for gene in markers:
            if gene in common_genes and gene not in selected_genes:
                selected_genes.append(gene)

        return selected_genes

    # select genes for svm
    selected_genes = select_genes(X_ref[train_idx], y[train_idx])
    selected_indices = [common_genes.get_loc(gene) for gene in selected_genes]

    X_train = X_ref[train_idx][:, selected_indices].toarray()
    X_test = X_ref[test_idx][:, selected_indices].toarray()

    print("number of selected genes:", len(selected_genes))
    print("marker genes found:", len([gene for gene in markers if gene in common_genes]))

    # old models
    # model = KNeighborsClassifier(n_neighbors=5, metric="cosine")
    # model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
    # model.fit(X_train, y[train_idx])
    # predicted = model.predict(X_test)

    # train lda model
    # model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    # model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=np.ones(5) / 5)

    # old svm with chi2 genes
    # model = LinearSVC(class_weight="balanced", C=0.1, max_iter=5000, random_state=42)

    # train svm using genes with higher expression
    model = LinearSVC(class_weight="balanced", C=0.1, max_iter=5000, random_state=42)
    model.fit(X_train, y[train_idx])

    # predict test cells
    predicted = model.predict(X_test)

    print("\nsvm validation results:")
    print("accuracy:", round(accuracy_score(y[test_idx], predicted), 3))
    print("balanced accuracy:", round(balanced_accuracy_score(y[test_idx], predicted), 3))
    print("macro f1:", round(f1_score(y[test_idx], predicted, average="macro"), 3))
    print(classification_report(y[test_idx], predicted, zero_division=0))

    # custom model using biological signatures from the paper
    signatures = {
        "P": ["Mki67", "Birc5", "Cks2", "Mcm2", "Mcm3", "E2f1", "Mad2l1", "Lig1"],
        "M": ["Tcf7", "Id3", "Ccr7", "Sell", "Slamf6", "Il7r", "Cxcr5"],
        "E": ["Cx3cr1", "Gzmb", "Prf1", "Zeb2", "Tbx21", "Klf2"],
        "Exh": ["Pdcd1", "Lag3", "Cd160", "Cd244", "Eomes", "Cxcr6", "Tox"]
    }

    # calculate expression scores for each biological signature
    def get_scores(X_data):
        scores = []

        for name in signatures:
            genes = signatures[name]
            indices = []

            for gene in genes:
                if gene in common_genes:
                    indices.append(common_genes.get_loc(gene))

            if len(indices) > 0:
                score = np.asarray(X_data[:, indices].mean(axis=1)).ravel()
            else:
                score = np.zeros(X_data.shape[0])

            scores.append(score)

        return np.column_stack(scores)

    # calculate avg signature profile for each phenotype
    def train_custom(X_scores, y_train):
        profiles = []

        for p in phenotypes:
            profile = X_scores[y_train == p].mean(axis=0)
            profiles.append(profile)

        return np.array(profiles)

    # compare each cell with the five signature profiles
    def predict_custom(X_scores, profiles):
        distances = []

        for profile in profiles:
            distance = np.sum((X_scores - profile)**2, axis=1)
            distances.append(distance)

        distances = np.column_stack(distances)
        best = np.argmin(distances, axis=1)

        return phenotypes[best]

    # calculate scores for training and test cells
    train_scores = get_scores(X_ref[train_idx])
    test_scores = get_scores(X_ref[test_idx])

    # standardize scores using training cells
    score_mean = train_scores.mean(axis=0)
    score_std = train_scores.std(axis=0) + 1e-10

    train_scores = (train_scores - score_mean) / score_std
    test_scores = (test_scores - score_mean) / score_std

    custom_model = train_custom(train_scores, y[train_idx])
    custom_predicted = predict_custom(test_scores, custom_model)

    print("\ncustom model validation results:")
    print("accuracy:", round(accuracy_score(y[test_idx], custom_predicted), 3))
    print("balanced accuracy:", round(balanced_accuracy_score(y[test_idx], custom_predicted), 3))
    print("macro f1:", round(f1_score(y[test_idx], custom_predicted, average="macro"), 3))
    print(classification_report(y[test_idx], custom_predicted, zero_division=0))

    # pearson nearest-centroid model
    # old model using average expression profiles
    # this version uses 2000 chi2 selected genes

    pearson_selector = SelectKBest(chi2, k=min(2000, X_ref.shape[1]))
    pearson_train = pearson_selector.fit_transform(X_ref[train_idx], y[train_idx])
    pearson_test = pearson_selector.transform(X_ref[test_idx])

    # calc mean expression profile for each phenotype
    def train_model(X_train, y_train):
        profiles = np.vstack([
            np.asarray(X_train[y_train == p].mean(axis=0)).ravel()
            for p in phenotypes
        ])

        # frequencies = np.array([
        #     np.mean(y_train == p)
        #     for p in phenotypes
        # ])

        profiles = profiles - profiles.mean(axis=1, keepdims=True)

        # return profiles, frequencies
        return profiles

    # compare cells with the five avg profiles
    def predict(X_data, pearson_model):
        # profiles, frequencies = model
        profiles = pearson_model

        # pearson correlation numerator
        correlations = np.asarray(X_data @ profiles.T)

        # denominator
        sums = np.asarray(X_data.sum(axis=1)).ravel()
        squares = np.asarray(X_data.multiply(X_data).sum(axis=1)).ravel()
        n_genes = X_data.shape[1]

        cell_norm = np.sqrt(np.maximum(squares - sums**2 / n_genes, 0))
        profile_norm = np.linalg.norm(profiles, axis=1)

        correlations = correlations / (cell_norm[:, None] * profile_norm[None, :] + 1e-10)

        # small penalty for common phenotypes
        # correlations = correlations - 0.005 * frequencies

        # pick phenotype with highest correlation
        best = np.argmax(correlations, axis=1)

        return phenotypes[best]

    pearson_model = train_model(pearson_train, y[train_idx])
    pearson_predicted = predict(pearson_test, pearson_model)

    print("\npearson validation results:")
    print("accuracy:", round(accuracy_score(y[test_idx], pearson_predicted), 3))
    print("balanced accuracy:", round(balanced_accuracy_score(y[test_idx], pearson_predicted), 3))
    print("macro f1:", round(f1_score(y[test_idx], pearson_predicted, average="macro"), 3))
    print(classification_report(y[test_idx], pearson_predicted, zero_division=0))

    # compare the three methods
    print("\nmodel comparison:")
    print("svm macro f1:", round(f1_score(y[test_idx], predicted, average="macro"), 3))
    print("custom macro f1:", round(f1_score(y[test_idx], custom_predicted, average="macro"), 3))
    print("pearson macro f1:", round(f1_score(y[test_idx], pearson_predicted, average="macro"), 3))

    # train again using all reference cells
    # X_ref = selector.fit_transform(X_ref, y)
    # X = selector.transform(X)
    #
    # final_model = train_model(X_ref, y)
    # cell_annotations["phenotype"] = predict(X, final_model)

    # old gene selection
    # selector.fit(X_ref, y)
    # selected_genes = list(common_genes[selector.get_support()])

    # old chi2 gene selection using all reference cells
    # selected_genes = []
    #
    # for p in phenotypes:
    #     binary_labels = (y == p).astype(int)
    #
    #     scores, _ = chi2(X_ref, binary_labels)
    #     scores = np.nan_to_num(scores)
    #
    #     best = np.argsort(scores)[-50:]
    #
    #     for gene in common_genes[best]:
    #         if gene not in selected_genes:
    #             selected_genes.append(gene)

    # choose which model to use for dataset 2
    # "svm", "custom" or "pearson"
    final_method = "svm"

    if final_method == "custom":
        # train custom model using all reference cells
        ref_scores = get_scores(X_ref)
        data_scores = get_scores(X)

        score_mean = ref_scores.mean(axis=0)
        score_std = ref_scores.std(axis=0) + 1e-10

        ref_scores = (ref_scores - score_mean) / score_std
        data_scores = (data_scores - score_mean) / score_std

        final_model = train_custom(ref_scores, y)
        cell_annotations["phenotype"] = predict_custom(data_scores, final_model)

    elif final_method == "pearson":
        # select genes again using all reference cells
        pearson_ref = pearson_selector.fit_transform(X_ref, y)
        pearson_data = pearson_selector.transform(X)

        final_model = train_model(pearson_ref, y)
        cell_annotations["phenotype"] = predict(pearson_data, final_model)

    else:
        # select genes again using all reference cells
        selected_genes = select_genes(X_ref, y)
        selected_indices = [common_genes.get_loc(gene) for gene in selected_genes]

        X_ref = X_ref[:, selected_indices].toarray()
        X = X[:, selected_indices].toarray()

        model.fit(X_ref, y)
        cell_annotations["phenotype"] = model.predict(X)

    return cell_annotations

**Please also submit the output CSV file with the annotations for Task 1b!**

In [58]:
task1b_cell_annotations = task1b_annotation(expression_data_2, expression_data_1, metadata_1)
task1b_cell_annotations.to_csv(os.path.join(OUTPUT_PATH, "task1b_cell_annotations.txt"), sep="\t")

# addition to the cell
print("predicted phenotype distribution:")
print(task1b_cell_annotations["phenotype"].value_counts())
display(task1b_cell_annotations.head())

number of selected genes: 194
marker genes found: 24

svm validation results:
accuracy: 0.577
balanced accuracy: 0.357
macro f1: 0.326
              precision    recall  f1-score   support

           E       0.51      0.58      0.54       914
         Exh       0.75      0.70      0.73      2281
           I       0.18      0.09      0.12       276
           M       0.13      0.38      0.19       154
           P       0.08      0.03      0.05       236

    accuracy                           0.58      3861
   macro avg       0.33      0.36      0.33      3861
weighted avg       0.59      0.58      0.58      3861


custom model validation results:
accuracy: 0.543
balanced accuracy: 0.3
macro f1: 0.284
              precision    recall  f1-score   support

           E       0.44      0.55      0.49       914
         Exh       0.73      0.68      0.70      2281
           I       0.09      0.04      0.06       276
           M       0.09      0.19      0.12       154
           P    

,phenotype
D_AAACGGGGTAGCGTAG,Exh
D_AAACGGGGTCTAGTCA,E
D_AAAGATGAGTATTGGA,P
D_AAAGATGTCATAGCAC,P
D_AAAGCAAAGGATCGCA,Exh


### Task 2

In [ ]:
cell_sequences = np.load(os.path.join(DATA_PATH, "tex_sampling.npy"))
metadata_3 = pd.read_csv(os.path.join(DATA_PATH, "expression_data_3_metadata.txt"), sep="\t", index_col=0)
expression_data_3 = import_mtx_expression_data(os.path.join(DATA_PATH, "expression_data_3.mtx"),
                                               os.path.join(DATA_PATH, "expression_data_3_barcodes.tsv"),
                                               os.path.join(DATA_PATH, "expression_data_3_genes.tsv"))

In [ ]:
def task2a_trajectory_analysis(sequences: np.ndarray, metadata: pd.DataFrame) -> Tuple[list, list]:
    """
    Identify an ordering of louvain clusters along the two trajectories
    :param sequences: numpy array of cell sequences (cell indices, corresponding to rows of metadata) of shape
    (n_sequences, n_steps)
    :param metadata: cell metadata with louvain clusters
    :return: orderings of louvain clusters along the two trajectories
    """
    upper_trajectory_louvain_ordering = []
    lower_trajectory_louvain_ordering = []

    #########################
    # INSERT YOUR CODE HERE #
    #########################

    return upper_trajectory_louvain_ordering, lower_trajectory_louvain_ordering

**Please also submit the output files with the cluster orderings for Task 2a!**

In [ ]:
task2a_upper_trajectory, task2a_lower_trajectory = task2a_trajectory_analysis(cell_sequences, metadata_3)
pd.DataFrame(task2a_upper_trajectory).to_csv(os.path.join(OUTPUT_PATH, "task2a_upper_trajectory.txt"), header=False)
pd.DataFrame(task2a_lower_trajectory).to_csv(os.path.join(OUTPUT_PATH, "task2a_lower_trajectory.txt"), header=False)

In [ ]:
def task2b_correlation_analysis(expression_data: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the correlation of each gene's expression with sample ID
    :param expression_data: expression matrix of shape (n_cells, n_genes)
    :param metadata: metadata containing sample IDs (column name sampleID)
    :return: data frame with correlation and p value for each gene
    """
    gene_correlations = pd.DataFrame(data=np.ones((expression_data.shape[1], 3)), index=expression_data.columns.values,
                                     columns=["correlation", "abs_correlation", "p_value"])

    #########################
    # INSERT YOUR CODE HERE #
    #########################

    return gene_correlations

**Please also submit the output CSV file with the correlations for Task 2b!**

In [ ]:
task2b_gene_correlations = task2b_correlation_analysis(expression_data_3, metadata_3)
task2b_gene_correlations.sort_values("abs_correlation", ascending=False).iloc[:10]\
    .to_csv(os.path.join(OUTPUT_PATH, "task2b_gene_correlations.txt"), sep="\t")

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(x=task2b_gene_correlations["correlation"], y=-np.log(task2b_gene_correlations["p_value"] + 1e-15), ax=ax)
ax.set_ylabel("-log p-value");